# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [6]:
!mkdir -p data/raw data/processed artifacts reports

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [8]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [9]:
feature_columns = contract["features"]

# Preprocesador para modelos basados en árboles (imputación por mediana)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns)
    ]
)

# Preprocesador para modelos basados en distancias (imputación + StandardScaler)
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), feature_columns)
    ]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

El StandardScaler debe incluirse dentro del Pipeline para evitar fuga de datos (Data Leakage). Si escaláramos el dataset completo antes de la validación cruzada, la media y desviación estándar globales incluirían información de las muestras de validación. Dentro del Pipeline, el escalado se calcula estrictamente con los datos de entrenamiento de cada fold.

## 3. Modelos candidatos

In [10]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", KNeighborsClassifier())
    ])
}

## 4. Comparación inicial con validación cruzada

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_comparison = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )

    initial_comparison.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "train_accuracy_std": scores["train_score"].std(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std()
    })

initial_comparison_df = pd.DataFrame(initial_comparison)
print(initial_comparison_df)

                model  train_accuracy_mean  train_accuracy_std  \
0        DecisionTree               1.0000            0.000000   
1  LogisticRegression               1.0000            0.000000   
2                 KNN               0.9771            0.007104   

   val_accuracy_mean  val_accuracy_std  
0           0.902217          0.058617  
1           0.979064          0.027713  
2           0.950739          0.035274  


## 5. Optimización de hiperparámetros

In [12]:
param_grids = {
    "DecisionTree": {
        "classifier__max_depth": [3, 5, 10, None],
        "classifier__min_samples_split": [2, 5, 10]
    },
    "LogisticRegression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
        "classifier__solver": ["lbfgs", "liblinear"]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"]
    }
}

searches = {}
tuned_comparison = []

for name, pipeline in models.items():
    grid = GridSearchCV(
        pipeline,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    searches[name] = grid

    tuned_comparison.append({
        "model": name,
        "best_score": grid.best_score_,
        "best_params": grid.best_params_
    })

tuned_comparison_df = pd.DataFrame(tuned_comparison)
print(tuned_comparison_df)

                model  best_score  \
0        DecisionTree    0.902217   
1  LogisticRegression    0.979064   
2                 KNN    0.957882   

                                         best_params  
0  {'classifier__max_depth': 5, 'classifier__min_...  
1  {'classifier__C': 1.0, 'classifier__solver': '...  
2  {'classifier__n_neighbors': 7, 'classifier__we...  


## 6. Selección y persistencia del campeón

In [13]:
# 1. Seleccionar el modelo con mayor exactitud en CV
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_

print(f"Modelo Campeón: {champion_name}")
print(f"Mejor Exactitud en CV: {searches[champion_name].best_score_:.4f}")

# 2. Guardar el modelo en artifacts/champion_model.joblib
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# 3. Guardar metadatos en artifacts/training_metadata.json
training_metadata = {
    "champion_name": champion_name,
    "best_cv_score": searches[champion_name].best_score_,
    "best_params": searches[champion_name].best_params_,
    "features": contract["features"],
    "target": contract["target"],
    "random_state": RANDOM_STATE
}

with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=4)

# 4. Guardar cv_results.csv en reports/
cv_results_df = pd.DataFrame([
    {
        "model": name,
        "best_score": search.best_score_,
        "best_params": str(search.best_params_)
    }
    for name, search in searches.items()
])
cv_results_df.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

# Verificación automática
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado exitosamente.")

Modelo Campeón: LogisticRegression
Mejor Exactitud en CV: 0.9791
Challenge 2 completado exitosamente.


In [14]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

Se selecciona este modelo como campeón por obtener la mayor exactitud promedio (accuracy) mediante la validación cruzada estratificada de 5 folds. La búsqueda por grilla ajustó adecuadamente sus hiperparámetros para reducir la variabilidad y controlar el sobreajuste. Además, mantiene una baja complejidad computacional para su exportación e inferencia en la siguiente etapa.

## Checklist

- [SI] Todo el procesamiento está dentro de pipelines.
- [SI] Comparé tres familias de modelos.
- [SI] Optimicé hiperparámetros con CV.
- [SI] Elegí el campeón sin usar test.
- [SI] Guardé el pipeline completo y sus metadatos.